In [1]:
"""
CarbonTatva AI — Model: XGBoost (Fixed for Distribution Shift)
===============================================================
Root cause of low accuracy found:
  Train mean price: Rs.5,332  (Dec 2022 - Dec 2024)
  Test mean price:  Rs.3,415  (Oct 2025 - Feb 2026)
  That is a 36% price level drop — model trained on expensive
  periods will always over-predict cheap ones.

Two fixes applied:
  Fix 1 — Relative target:
    Instead of predicting absolute price (Rs/MWh),
    predict price as a RATIO to recent 7-day moving average.
    ratio = price / ma7d
    After prediction: price = predicted_ratio x ma7d
    This removes the long-term level drift from the target.

  Fix 2 — Rolling 18-month training window:
    Only use the most recent 18 months for training.
    This ensures the model learns from price levels
    closer to the current test period.

Run:
    python model_xgboost.py

Output: outputs_xgboost/
"""

import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
os.makedirs('outputs_xgboost', exist_ok=True)

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
DATA_FILE      = 'final_merged_data2.xlsx'
MARKETS        = ['GDAM', 'DAM', 'RTM']
SLOTS_DAY      = 96
TRAIN_END      = '2024-12-31'
VAL_END        = '2025-09-30'
ROLLING_MONTHS = 18
SEED           = 42
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────
def build_features(df):
    df = df.copy()
    d = pd.to_datetime(df['Delivery Date'], dayfirst=True)
    t = df['Time Period'].str.split('-').str[0]
    df['datetime'] = pd.to_datetime(d.astype(str) + ' ' + t)
    df = df.sort_values('datetime').reset_index(drop=True)

    df.drop(columns=['apparent_temperature (°C)',
                     'sunshine_duration (s)',
                     'precipitation (mm)'], errors='ignore', inplace=True)

    # Temporal
    df['hour']    = df['datetime'].dt.hour
    df['dow']     = df['datetime'].dt.dayofweek
    df['month']   = df['datetime'].dt.month
    df['quarter'] = df['datetime'].dt.quarter

    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)
    df['is_weekend']   = (df['dow'] >= 5).astype(int)
    df['is_morning']   = ((df['hour'] >= 6)  & (df['hour'] <= 9)).astype(int)
    df['is_evening']   = ((df['hour'] >= 18) & (df['hour'] <= 21)).astype(int)
    df['is_solar_hrs'] = ((df['hour'] >= 8)  & (df['hour'] <= 16)).astype(int)

    try:
        import holidays
        india_h = holidays.India(years=range(2022, 2027))
        df['holiday_flag'] = df['datetime'].dt.date.apply(
            lambda d: 1 if d in india_h else 0)
    except ImportError:
        df['holiday_flag'] = 0

    # Autoregressive
    p = df['Price (Rs./MWh)']
    df['price_lag96']   = p.shift(96)
    df['price_lag192']  = p.shift(192)
    df['price_lag288']  = p.shift(288)
    df['price_lag672']  = p.shift(672)
    df['price_lag1344'] = p.shift(1344)

    df['ma1d']  = p.shift(1).rolling(96).mean()
    df['ma3d']  = p.shift(1).rolling(288).mean()
    df['ma7d']  = p.shift(1).rolling(672).mean()
    df['ma14d'] = p.shift(1).rolling(1344).mean()
    df['ma30d'] = p.shift(1).rolling(2880).mean()

    df['std3d'] = p.shift(1).rolling(288).std()
    df['std7d'] = p.shift(1).rolling(672).std()

    df['price_diff1d'] = p.shift(96)   - p.shift(192)
    df['price_diff1w'] = p.shift(672)  - p.shift(1344)
    df['price_diff2w'] = p.shift(1344) - p.shift(2016)

    df['max_prev_day']   = p.shift(1).rolling(96).max()
    df['min_prev_day']   = p.shift(1).rolling(96).min()
    df['range_prev_day'] = df['max_prev_day'] - df['min_prev_day']

    # Price level / regime features — tells model current price regime
    df['price_level_ratio_7d']  = df['price_lag96'] / (df['ma7d']  + 1e-6)
    df['price_level_ratio_30d'] = df['price_lag96'] / (df['ma30d'] + 1e-6)
    df['rel_lag96']             = df['price_lag96']  / (df['ma7d']  + 1e-6)
    df['rel_lag672']            = df['price_lag672'] / (df['ma30d'] + 1e-6)

    # Weather
    df['temp']    = df['temperature_2m (°C)']
    df['temp_sq'] = df['temp'] ** 2
    df['radiation']            = df['shortwave_radiation (W/m²)']
    df['radiation_x_hour_sin'] = df['radiation'] * df['hour_sin']
    df['radiation_x_is_solar'] = df['radiation'] * df['is_solar_hrs']
    df['wind_speed']  = df['wind_speed_100m (km/h)']
    df['cloud_cover'] = df['cloud_cover (%)']
    df['humidity']    = df['relative_humidity_2m (%)']
    df['rain']        = df['rain (mm)']
    df['temp_x_evening']    = df['temp'] * df['is_evening']
    df['temp_x_morning']    = df['temp'] * df['is_morning']
    df['temp_x_is_weekend'] = df['temp'] * df['is_weekend']

    # Target: price relative to recent 7-day moving average
    df['target_ratio'] = p / (df['ma7d'] + 1e-6)

    return df


FEATURES = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'dow_sin', 'dow_cos', 'quarter',
    'is_weekend', 'holiday_flag',
    'is_morning', 'is_evening', 'is_solar_hrs',
    'price_lag96', 'price_lag192', 'price_lag288',
    'price_lag672', 'price_lag1344',
    'ma1d', 'ma3d', 'ma7d', 'ma14d', 'ma30d',
    'std3d', 'std7d',
    'price_diff1d', 'price_diff1w', 'price_diff2w',
    'max_prev_day', 'min_prev_day', 'range_prev_day',
    'price_level_ratio_7d', 'price_level_ratio_30d',
    'rel_lag96', 'rel_lag672',
    'temp', 'temp_sq',
    'radiation', 'radiation_x_hour_sin', 'radiation_x_is_solar',
    'wind_speed', 'cloud_cover', 'humidity', 'rain',
    'temp_x_evening', 'temp_x_morning', 'temp_x_is_weekend',
]

# ─────────────────────────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, label=''):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    pct  = mae / np.mean(y_true) * 100
    print(f"    {label:22s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  "
          f"MAPE={mape:.2f}%  (MAE={pct:.1f}% of mean)")
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def save_pred_plot(y_true, y_pred, title, path):
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27')
        ax.tick_params(colors='#aaa')
    n = min(800, len(y_true))
    axes[0].plot(y_true[:n], color='#aaa', lw=0.8, label='Actual')
    axes[0].plot(y_pred[:n], color='#e67e22', lw=0.8, label='Predicted')
    axes[0].set_title(f'{title} — Test Set', color='white', fontsize=12)
    axes[0].legend(fontsize=9)
    axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')
    n2 = SLOTS_DAY * 5
    axes[1].plot(y_true[:n2], color='#aaa', lw=1.5, label='Actual')
    axes[1].plot(y_pred[:n2], color='#e67e22', lw=1.5, ls='--', label='Predicted')
    axes[1].set_title(f'{title} — First 5 Days Zoomed', color='white', fontsize=12)
    axes[1].legend(fontsize=9)
    axes[1].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[2].scatter(y_true[:3000], y_pred[:3000], s=2, alpha=0.3, color='#e67e22')
    lim = max(y_true[:3000].max(), y_pred[:3000].max())
    axes[2].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect')
    axes[2].set_title(f'{title} — Actual vs Predicted Scatter', color='white', fontsize=12)
    axes[2].set_xlabel('Actual Price', color='#aaa')
    axes[2].set_ylabel('Predicted Price', color='#aaa')
    axes[2].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()


def save_monthly_plot(test_df, test_pred, mkt, path):
    test_copy = test_df.copy()
    test_copy['predicted'] = test_pred
    test_copy['abs_error'] = np.abs(
        test_copy['Price (Rs./MWh)'] - test_pred)
    test_copy['ym'] = test_copy['datetime'].dt.to_period('M')
    monthly = test_copy.groupby('ym').agg(
        actual=('Price (Rs./MWh)', 'mean'),
        predicted=('predicted', 'mean'),
        mae=('abs_error', 'mean')).round(1)

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27')
        ax.tick_params(colors='#aaa')
    months = monthly.index.astype(str)
    x = np.arange(len(months))
    w = 0.35
    axes[0].bar(x - w/2, monthly['actual'],    w,
                color='#aaa', alpha=0.8, label='Actual Mean')
    axes[0].bar(x + w/2, monthly['predicted'], w,
                color='#e67e22', alpha=0.8, label='Predicted Mean')
    axes[0].set_title(f'{mkt} — Monthly Mean Price: Actual vs Predicted',
                      color='white')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(months, rotation=30, ha='right', color='#aaa')
    axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[0].legend(fontsize=9)

    axes[1].bar(x, monthly['mae'], color='#e74c3c', alpha=0.85)
    axes[1].set_title(f'{mkt} — Monthly MAE', color='white')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(months, rotation=30, ha='right', color='#aaa')
    axes[1].set_ylabel('MAE (Rs./MWh)', color='#aaa')

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()
    print(f"\n  Monthly test breakdown:")
    print(monthly.to_string())


# ─────────────────────────────────────────────────────────────────
# MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("MODEL: XGBoost (Fixed)")
print("=" * 60)
print(f"Root cause: 36% price drop from train to test period")
print(f"Fix: predict price/MA7d ratio + rolling {ROLLING_MONTHS}mo window\n")

print(f"Loading {DATA_FILE}...")
sheets      = pd.read_excel(DATA_FILE, sheet_name=None)
req_feats   = FEATURES + ['target_ratio', 'ma7d', 'Price (Rs./MWh)']
all_metrics = {}

for mkt in MARKETS:
    print(f"\n{'─'*55}")
    print(f"Market: {mkt}")
    print('─' * 55)

    df = build_features(sheets[mkt])
    df = df.dropna(subset=req_feats).copy()
    df = df[np.isfinite(df['target_ratio'])].copy()

    full_train = df[df['datetime'] <= TRAIN_END]
    val        = df[(df['datetime'] > TRAIN_END) &
                    (df['datetime'] <= VAL_END)]
    test       = df[df['datetime'] > VAL_END]

    print(f"  Full train: {len(full_train):,} | "
          f"Val: {len(val):,} | Test: {len(test):,}")
    print(f"  Train price mean: Rs.{full_train['Price (Rs./MWh)'].mean():.0f}  "
          f"Test price mean: Rs.{test['Price (Rs./MWh)'].mean():.0f}")

    # Rolling window — last 18 months only
    rolling_start = pd.Timestamp(TRAIN_END) - pd.DateOffset(
        months=ROLLING_MONTHS)
    train = full_train[full_train['datetime'] >= rolling_start]
    print(f"  Rolling train: {len(train):,} rows "
          f"(from {train['datetime'].min().date()})")

    X_tr = train[FEATURES].values
    X_va = val[FEATURES].values
    X_te = test[FEATURES].values

    y_tr_ratio = train['target_ratio'].values
    y_va_ratio = val['target_ratio'].values

    ma7d_va   = val['ma7d'].values
    ma7d_te   = test['ma7d'].values
    y_va_abs  = val['Price (Rs./MWh)'].values
    y_te_abs  = test['Price (Rs./MWh)'].values

    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_va_s  = scaler.transform(X_va)
    X_te_s  = scaler.transform(X_te)

    dtrain = xgb.DMatrix(X_tr_s, label=y_tr_ratio, feature_names=FEATURES)
    dval   = xgb.DMatrix(X_va_s, label=y_va_ratio, feature_names=FEATURES)
    dtest  = xgb.DMatrix(X_te_s,                   feature_names=FEATURES)

    params = {
        'objective':         'reg:squarederror',
        'eval_metric':       'mae',
        'max_depth':         7,
        'learning_rate':     0.02,
        'subsample':         0.85,
        'colsample_bytree':  0.8,
        'colsample_bylevel': 0.8,
        'min_child_weight':  30,
        'gamma':             0.05,
        'reg_alpha':         0.1,
        'reg_lambda':        1.0,
        'tree_method':       'hist',
        'max_bin':           256,
        'seed':              SEED,
        'verbosity':         0,
    }

    print(f"  Training on ratio target...")
    evals_result = {}
    model = xgb.train(
        params, dtrain,
        num_boost_round=3000,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=80,
        evals_result=evals_result,
        verbose_eval=300,
    )
    best_iter = model.best_iteration
    print(f"  Best iteration: {best_iter}")

    # Predict ratio → multiply by ma7d to get absolute price
    val_ratio_pred  = model.predict(dval,  iteration_range=(0, best_iter))
    test_ratio_pred = model.predict(dtest, iteration_range=(0, best_iter))
    val_pred_abs    = np.clip(val_ratio_pred  * ma7d_va, 0, 20000)
    test_pred_abs   = np.clip(test_ratio_pred * ma7d_te, 0, 20000)

    print(f"\n  Results:")
    m_val  = compute_metrics(y_va_abs, val_pred_abs,  'Validation')
    m_test = compute_metrics(y_te_abs, test_pred_abs, 'Test')
    all_metrics[mkt] = {'val': m_val, 'test': m_test}

    # Naive baseline comparison
    naive_mae = np.mean(np.abs(y_te_abs - test['price_lag96'].values))
    print(f"    {'Naive lag96 baseline':22s}  MAE={naive_mae:7.1f}  "
          f"(XGB beats naive by "
          f"{(naive_mae - m_test['mae']) / naive_mae * 100:.1f}%)")

    # Training curve
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0f1117')
    ax.set_facecolor('#1a1d27')
    ax.plot(evals_result['train']['mae'], color='#e74c3c', lw=1, label='Train MAE')
    ax.plot(evals_result['val']['mae'],   color='#27ae60', lw=1, label='Val MAE')
    ax.axvline(best_iter, color='yellow', lw=1.5, ls='--',
               label=f'Best iter={best_iter}')
    ax.set_title(f'XGBoost Training Curve — {mkt}', color='white')
    ax.set_xlabel('Round', color='#aaa')
    ax.set_ylabel('MAE (ratio target)', color='#aaa')
    ax.legend(fontsize=9)
    ax.tick_params(colors='#aaa')
    plt.tight_layout()
    plt.savefig(f'outputs_xgboost/xgb_loss_{mkt}.png',
                dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    # Feature importance
    imp = model.get_score(importance_type='gain')
    if imp:
        names = list(imp.keys())
        vals  = list(imp.values())
        idx   = np.argsort(vals)[-25:]
        fig, ax = plt.subplots(figsize=(11, 9))
        fig.patch.set_facecolor('#0f1117')
        ax.set_facecolor('#1a1d27')
        ax.barh([names[i] for i in idx], [vals[i] for i in idx],
                color='#e67e22', alpha=0.85)
        ax.set_title(f'Feature Importance — {mkt}', color='white', fontsize=12)
        ax.tick_params(colors='#aaa')
        ax.set_xlabel('Gain', color='#aaa')
        plt.tight_layout()
        plt.savefig(f'outputs_xgboost/xgb_importance_{mkt}.png',
                    dpi=150, bbox_inches='tight', facecolor='#0f1117')
        plt.close()

    save_pred_plot(y_te_abs, test_pred_abs,
                   f'XGBoost — {mkt}',
                   f'outputs_xgboost/xgb_pred_{mkt}.png')

    save_monthly_plot(test, test_pred_abs, mkt,
                      f'outputs_xgboost/xgb_monthly_{mkt}.png')

    model.save_model(f'outputs_xgboost/xgb_{mkt}.json')
    with open(f'outputs_xgboost/xgb_scaler_{mkt}.pkl', 'wb') as f:
        pickle.dump({'scaler': scaler, 'features': FEATURES}, f)
    print(f"\n  Saved: outputs_xgboost/xgb_{mkt}.json")


# ─────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("SUMMARY — XGBoost (Fixed)")
print('=' * 60)
rows = []
for mkt, res in all_metrics.items():
    for split, m in res.items():
        rows.append({'Market': mkt, 'Split': split,
                     'MAE': round(m['mae'], 1),
                     'RMSE': round(m['rmse'], 1),
                     'MAPE%': round(m['mape'], 2)})
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv('outputs_xgboost/metrics.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('XGBoost — Test MAE by Market', color='white', fontsize=14)
for ax, mkt in zip(axes, MARKETS):
    ax.set_facecolor('#1a1d27')
    if mkt in all_metrics:
        mae = all_metrics[mkt]['test']['mae']
        ax.bar([mkt], [mae], color='#e67e22', alpha=0.85)
        ax.text(0, mae + 5, f'Rs.{mae:.0f}', ha='center',
                color='white', fontsize=12, fontweight='bold')
    ax.set_title(mkt, color='white')
    ax.set_ylabel('MAE (Rs./MWh)', color='#aaa')
    ax.tick_params(colors='#aaa')
plt.tight_layout()
plt.savefig('outputs_xgboost/summary_chart.png',
            dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.close()
print("\nDone! Check outputs_xgboost/")

MODEL: XGBoost (Fixed)
Root cause: 36% price drop from train to test period
Fix: predict price/MA7d ratio + rolling 18mo window

Loading final_merged_data.xlsx...

───────────────────────────────────────────────────────
Market: GDAM
───────────────────────────────────────────────────────
  Full train: 101,185 | Val: 26,208 | Test: 14,591
  Train price mean: Rs.5843  Test price mean: Rs.3984
  Rolling train: 52,801 rows (from 2023-06-30)
  Training on ratio target...
[0]	train-mae:0.34509	val-mae:0.42480
[300]	train-mae:0.07308	val-mae:0.12996
[501]	train-mae:0.06518	val-mae:0.12943
  Best iteration: 421

  Results:
    Validation              MAE=  594.1  RMSE=  921.3  MAPE=16.87%  (MAE=12.6% of mean)
    Test                    MAE=  388.1  RMSE=  680.2  MAPE=11.04%  (MAE=9.7% of mean)
    Naive lag96 baseline    MAE=  420.8  (XGB beats naive by 7.8%)

  Monthly test breakdown:
         actual  predicted    mae
ym                               
2025-09  4185.4     3907.3  485.6
2025-1